# **Sesión 1 - Modelos de Vectores Autorregresivos**
## _B) Pronósticos incondicionales y condicionales_

El modelo macrofiscal del
[tutorial de `MacroPy`](https://github.com/RenatoVassallo/MacroPy/blob/main/tutorials/tutorial_bvar.ipynb):
crecimiento interanual de los precios de exportación ($x_t$), del PBI ($y_t$) y de los ingresos
fiscales del gobierno general ($r_t$), en un VAR(2) trimestral con $\mathbf{y}_t = (x_t,\ y_t,\ r_t)'$.
Todo este notebook usa datos **hasta 2019**.

1. **Estimación** con datos de 2002Q2 a 2017Q4.
2. **La densidad predictiva a mano**, con la forma companion y los choques sorteados con Cholesky,
   contrastada con `MacroPy` y con el error cuadrático medio (ECM).
3. **Pronóstico recursivo 2018-2019** contra lo que efectivamente ocurrió.
4. **Pronósticos condicionales**: la matriz $A$ de Waggoner y Zha armada a mano y dos escenarios
   externos para los precios de exportación.

*Tiempo de corrida: menos de un minuto.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from MacroPy import BayesianVAR
from MacroPy.plots_uc import plot_fan_chart
from MacroPy.plots_kalman import set_bse_style
from macrofiscal_data import load_levels, yoy_growth, quarter

set_bse_style()
levels, source = load_levels()
growth = yoy_growth(levels).loc[:"2019-12-01"]   # pre-pandemic sample
names = ["x", "y", "r"]
col = {c: i for i, c in enumerate(names)}
train = growth.loc[:"2017-12-01"]
actual = growth.loc["2018-03-01":"2019-12-01"]
n, p, h = 3, 2, 8                                  # variables, lags, horizon
dates = actual.index
titles = {"x": "Precios de exportación (x)", "y": "PBI (y)",
          "r": "Ingresos fiscales (r)"}

span = f"{quarter(train.index[0])} a {quarter(train.index[-1])}"
print(f"Fuente: {source}")
print(f"Estimación: {span} ({len(train)} trimestres); "
      "pronóstico: 2018Q1 a 2019Q4")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), constrained_layout=True)
for ax, c in zip(axes, names):
    ax.plot(growth.index, growth[c], color="#1F3D5C", lw=1.3)
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.axvspan(pd.Timestamp("2018-01-01"), pd.Timestamp("2019-12-31"),
               color="#F0E0D6", alpha=0.8, lw=0)
    ax.axhline(0, color="gray", lw=0.7)
    ax.set_title(f"{titles[c]}, var. % interanual", fontsize=10)
fig.suptitle("Modelo macrofiscal: estimación 2002-2017 y ventana "
             "de pronóstico 2018-2019", fontsize=11, weight="bold")
plt.show()

## 1. El modelo

VAR(2) con prior de Minnesota e inversa-Wishart, como en la especificación "realista" del tutorial:
$\delta_i = 0.5$, $\lambda_1 = 2$ y $\lambda_2 = 1$. Las tres ecuaciones incluyen los rezagos de
las tres variables; restringir la ecuación de $x$ (exogeneidad de bloque) es tema de la sesión 2.

In [ ]:
priors = {"mn_mean": 0.5, "lambda1": 2, "lambda2": 1}
bvar = BayesianVAR(train, lags=p, prior_type=2, prior_params=priors,
                   post_draws=8000, burnin=0.5, fhor=h, seed=42)
bvar.sample_posterior()
k = bvar.ncoeff_eq
draws_b = np.asarray(bvar.beta_draws)
draws_S = np.asarray(bvar.Sigma_draws)
M = len(draws_b)

print(f"{M} extracciones de la posterior; coeficientes por ecuación: {k}")

## 2. La densidad predictiva, a mano

`MacroPy` guarda cada extracción ecuación por ecuación, con la constante al final. La función
`to_book_B` la lleva a la matriz $B$ ($k \times n$) de las diapositivas, con la constante en la
primera fila, y `companion` arma $F$ y $\tilde c$. Para cada extracción $m$:

1. tomar $b^{(m)}$ y $\Sigma^{(m)}$, y armar $F^{(m)}$ y $\tilde c^{(m)}$;
2. sortear $\mathbf{u}^{(m)}_{T+j} = S^{(m)}\varepsilon_{T+j}$, con $S^{(m)}$ el factor de Cholesky
   de $\Sigma^{(m)}$ y $\varepsilon_{T+j} \sim N(0, I_n)$;
3. iterar $\tilde{\mathbf{y}}^{(m)}_{T+j} = \tilde c^{(m)} + F^{(m)}\tilde{\mathbf{y}}^{(m)}_{T+j-1} + \tilde{\mathbf{u}}^{(m)}_{T+j}$
   desde el mismo $\tilde{\mathbf{y}}_T$ observado.

Dada una extracción, sortear un choque nuevo en cada trimestre **acumula** el error cuadrático medio,
$\operatorname{ECM}^{(m)}_j = \operatorname{ECM}^{(m)}_{j-1} + \Psi_{j-1}\Sigma^{(m)}\Psi_{j-1}'$ con
$\Psi_i = JF^iJ'$, igual que en el pronóstico por MCO. Entre extracciones cambia además la media, así que
$\operatorname{Var}(\mathbf{y}_{T+j} \mid Y) = \mathbb E_m\big[\operatorname{ECM}^{(m)}_j\big] + \operatorname{Var}_m\big(\hat{\mathbf{y}}^{(m)}_{T+j}\big)$:
choques futuros más incertidumbre de los parámetros. La columna "solo ECM" muestra el primer término.
La Cholesky es solo una forma de sortear de $N(0, \Sigma^{(m)})$: cualquier $S$ con $SS' = \Sigma^{(m)}$
da la misma densidad. Comparamos desviaciones estándar por horizonte: comparar percentiles casilla
por casilla confunde el error de Monte Carlo con diferencias reales.

In [ ]:
J = np.hstack([np.eye(n), np.zeros((n, n * (p - 1)))])

def to_book_B(b_draw):
    """MacroPy draw (by equation, constant last) to B (k x n)."""
    coef = b_draw.reshape(n, k)              # row i: equation i
    return np.vstack([coef[:, -1], coef[:, :-1].T])

def companion(B):
    """Companion matrix F and stacked constant c_tilde."""
    F = np.zeros((n * p, n * p))
    F[:n, :] = B[1:, :].T
    F[n:, :-n] = np.eye(n * (p - 1))
    c_tilde = np.zeros(n * p)
    c_tilde[:n] = B[0]
    return F, c_tilde

history = train.to_numpy()
state = np.concatenate([history[-1 - i] for i in range(p)])

rng = np.random.default_rng(0)
paths = np.zeros((M, h, n))                  # with future shocks
means = np.zeros((M, h, n))                  # without shocks
ecm = np.zeros((M, h, n))
for m in range(M):
    F, c_tilde = companion(to_book_B(draws_b[m]))
    S = np.linalg.cholesky(draws_S[m])
    y_sim, y_mean = state.copy(), state.copy()
    F_j, acc = np.eye(n * p), np.zeros((n, n))
    for j in range(h):
        u_tilde = np.zeros(n * p)
        u_tilde[:n] = S @ rng.standard_normal(n)       # u = S eps
        y_sim = c_tilde + F @ y_sim + u_tilde
        paths[m, j] = J @ y_sim,

        y_mean = c_tilde + F @ y_mean
        means[m, j] = J @ y_mean
        
        Psi = J @ F_j @ J.T
        acc = acc + Psi @ draws_S[m] @ Psi.T
        ecm[m, j] = np.diag(acc)
        F_j = F_j @ F

forecast_draws = bvar.forecast(fhor=h, plot_forecast=False)["forecast_draws"]
sd_formula = np.sqrt(ecm.mean(0) + means.var(0))
sd_table = pd.concat(
    {"a mano": pd.DataFrame(paths.std(0), columns=names),
     "MacroPy": pd.DataFrame(forecast_draws.std(0), columns=names),
     "fórmula": pd.DataFrame(sd_formula, columns=names),
     "solo ECM": pd.DataFrame(np.sqrt(ecm.mean(0)), columns=names)},
    axis=1)
sd_table.index = [f"h={j}" for j in range(1, h + 1)]
print("Desviación estándar predictiva por horizonte (puntos porcentuales):")
sd_table.round(2)

Las tres primeras columnas coinciden dentro del error de Monte Carlo: el algoritmo de las
diapositivas es exactamente lo que hace `MacroPy`. La desviación crece con el horizonte porque el ECM
se acumula: para $x$ pasa de 8.5 puntos a un trimestre a 21.6 a dos años. La distancia entre
"fórmula" y "solo ECM" es la incertidumbre de los parámetros, que agrega alrededor de 5 % a la
desviación estándar en las tres variables.

## 3. Pronóstico 2018-2019 contra lo observado

Bandas al **68 %** y al **95 %**, calculadas con percentiles de las trayectorias. La de 68 % es el
rango "probable", cerca de una desviación estándar a cada lado; la de 95 % muestra el riesgo de cola.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), constrained_layout=True)
for ax, c in zip(axes, names):
    plot_fan_chart(train[c], forecast_draws[:, :, col[c]], dates,
                   bands=(0.95, 0.68), history_from="2012-03-01",
                   hline=0, ax=ax, color="#1F3D5C",
                   center_label="mediana",
                   history_label="observado hasta 2017",
                   title=f"{titles[c]}, var. % interanual")
    ax.plot(actual.index, actual[c], "o", color="#B0413E", ms=4.5,
            label="observado 2018-2019")
    ax.legend(fontsize=7, loc="lower left")
plt.show()

q = {lvl: np.percentile(forecast_draws, lvl, axis=0)
     for lvl in (2.5, 16, 50, 84, 97.5)}
obs = actual.to_numpy()
evaluation = pd.DataFrame({
    "dentro de la banda al 68 %": ((obs >= q[16]) & (obs <= q[84])).mean(0),
    "dentro de la banda al 95 %": ((obs >= q[2.5]) & (obs <= q[97.5])).mean(0),
    "RECM de la mediana (pp)": np.sqrt(((q[50] - obs) ** 2).mean(0)),
    "mediana 2019Q4": q[50][-1], "observado 2019Q4": obs[-1]},
    index=names)
evaluation.round(2)

La lectura honesta. Los precios de exportación caen dentro de la banda al 68 % en seis de ocho
trimestres y siempre dentro de la de 95 %; los ingresos fiscales, en siete de ocho, aunque ese octavo
queda fuera incluso de la banda al 95 %. El
PBI observado queda **por debajo** de la mediana en 2019: el modelo, estimado con el ciclo 2002-2017,
esperaba que la economía volviera a crecer cerca de 5 %, y no anticipó la desaceleración. La banda al
95 % sí contiene lo que pasó, y para eso está.

## 4. Pronósticos condicionales

Con $\mathbf{u}_t = S\varepsilon_t$ ($S$ de Cholesky, con $x$ primero), el futuro es el pronóstico sin
choques más el efecto de los choques estructurales:
$\mathbf{y}_{T+j} = \hat{\mathbf{y}}_{T+j|T} + \sum_{i<j}\Theta_i\,\varepsilon_{T+j-i}$, con $\Theta_i = \Psi_i S$.
Apilando los $h$ trimestres, las filas de $x$ forman la restricción
$A\,\varepsilon_{T+1:T+h} = \bar{\mathbf{x}} - \hat{\mathbf{x}}$: senda impuesta menos pronóstico sin choques.
Waggoner y Zha (1999) muestran que los choques que cumplen la restricción se distribuyen
$N\big(A^{+}(\bar{\mathbf{x}} - \hat{\mathbf{x}}),\ I - A^{+}A\big)$.

Primero armamos $A$ para $h = 2$ con una extracción, para ver su forma. Con $x$ primero en la Cholesky,
en $T+1$ solo el choque externo, $\varepsilon_1$, mueve a $x$; desde $T+2$ los choques domésticos
también llegan a $x$ por los rezagos.

In [ ]:
def responses(B, Sigma, horizon):
    """Theta_i = Psi_i S for i < horizon, with S the Cholesky factor."""
    F, _ = companion(B)
    S = np.linalg.cholesky(Sigma)
    out, F_i = [], np.eye(n * p)
    for _ in range(horizon):
        out.append(J @ F_i @ J.T @ S)
        F_i = F_i @ F
    return np.array(out)

def restriction_matrix(Theta, var, horizon):
    """Rows of the stacked block lower-triangular Theta for one variable."""
    A = np.zeros((horizon, n * horizon))
    for j in range(horizon):
        for i in range(j + 1):
            A[j, i * n:(i + 1) * n] = Theta[j - i][var]
    return A

Theta_2 = responses(to_book_B(draws_b[0]), draws_S[0], 2)
A_2 = restriction_matrix(Theta_2, col["x"], 2)
shock_labels = [f"ε{v},T+{j}" for j in (1, 2) for v in (1, 2, 3)]
pd.DataFrame(A_2, index=["x, T+1", "x, T+2"],
             columns=shock_labels).round(4)

### Dos escenarios externos

- **Adverso**: una caída con la forma de la crisis financiera internacional. Tomamos la trayectoria
  observada de $x$ entre 2008Q3 y 2009Q3, la escalamos para que toque fondo en $-25\,\%$ y la llevamos
  a cero en los últimos trimestres, sin el rebote por efecto base.
- **Favorable**: $x$ crece 10 puntos por encima de su trayectoria base.

In [ ]:
base_p = np.median(forecast_draws[:, :, col["x"]], axis=0)
crisis = growth.loc["2008-09-01":"2009-09-01", "x"].to_numpy()
fall = crisis * (-25 / crisis.min())
path_adverse = np.r_[fall, np.linspace(fall[-1], 0, 4)[1:]]
path_favorable = base_p + 10
print("x observado 2008Q3-2009Q3:", crisis.round(1))
print("Senda adversa:  ", path_adverse.round(1))
print("Senda favorable:", path_favorable.round(1))

imposed = {"favorable (+10 pp)": path_favorable,
           "adverso (crisis 2008-2009)": path_adverse}
scenarios = {"base": forecast_draws}
for name, path in imposed.items():
    cond, _ = bvar.conditional_forecast({"x": list(path)}, fhor=h,
                                        plot_forecast=False)
    scenarios[name] = cond

### El mismo cálculo, a mano

Con `shock_uncertainty=False`, `MacroPy` usa solo la media condicional de los choques,
$A^{+}(\bar{\mathbf{x}} - \hat{\mathbf{x}})$. La replicamos extracción por extracción con $A$ para
$h = 8$ y las trayectorias sin choques de la sección 2, y medimos cuánto se mueve cada choque.

In [ ]:
cond_mean, _ = bvar.conditional_forecast(
    {"x": list(path_adverse)}, fhor=h, plot_forecast=False,
    shock_uncertainty=False)

by_hand = np.zeros((M, h, n))
size_by_shock = np.zeros((M, n))
for m in range(M):
    Theta = responses(to_book_B(draws_b[m]), draws_S[m], h)
    A = restriction_matrix(Theta, col["x"], h)
    eps = np.linalg.pinv(A) @ (path_adverse - means[m, :, col["x"]])
    size_by_shock[m] = np.abs(eps.reshape(h, n)).sum(0)
    for j in range(h):
        by_hand[m, j] = means[m, j] + sum(
            Theta[j - i] @ eps[i * n:(i + 1) * n] for i in range(j + 1))

print(f"max |a mano - MacroPy| = {np.abs(by_hand - cond_mean).max():.1e}")
mean_size = size_by_shock.mean(0)
print("Choques condicionales medios: |ε| sumado sobre h y participación")
pd.DataFrame({"|ε|": mean_size, "participación": mean_size / mean_size.sum()},
             index=["externo (x)", "PBI (y)", "ingresos (r)"]).round(3)

In [ ]:
colors = {"base": "#6B6B6B", "favorable (+10 pp)": "#2E7D32",
          "adverso (crisis 2008-2009)": "#B0413E"}
fig, axes = plt.subplots(1, 3, figsize=(14, 3.9), constrained_layout=True)
for ax, c in zip(axes, names):
    recent = growth[c].loc["2014-03-01":"2017-12-01"]
    ax.plot(recent.index, recent, color="black", lw=1.2, label="observado")
    for name, draws in scenarios.items():
        z = draws[:, :, col[c]]
        ax.fill_between(dates, np.percentile(z, 16, 0),
                        np.percentile(z, 84, 0), color=colors[name],
                        alpha=0.15, lw=0)
        ax.plot(dates, np.median(z, 0), color=colors[name], lw=2,
                label=name)
    ax.axhline(0, color="gray", lw=0.7)
    ax.set_title(f"{titles[c]}, var. % interanual", fontsize=10)
axes[0].legend(fontsize=7, loc="lower left")
plt.show()

effects = {}
for name in imposed:
    d = np.median(scenarios[name], 0) - np.median(forecast_draws, 0)
    iy, ir = col["y"], col["r"]
    effects[name] = {
        "y, promedio 2018-19 (pp)": d[:, iy].mean(),
        "y, máximo efecto (pp)": d[np.argmax(np.abs(d[:, iy])), iy],
        "r, promedio 2018-19 (pp)": d[:, ir].mean(),
        "r, máximo efecto (pp)": d[np.argmax(np.abs(d[:, ir])), ir]}
effects = pd.DataFrame(effects).T
effects["razón r / y (promedio)"] = (effects["r, promedio 2018-19 (pp)"]
                                     / effects["y, promedio 2018-19 (pp)"])
print("Diferencia de medianas respecto del escenario base:")
effects.round(2)

Tres lecturas. Primero, la matriz $A$: en $T+1$ solo el choque externo mueve a $x$, pero desde $T+2$
los choques domésticos también llegan por los rezagos, y $A^{+}$ los usa. En el escenario adverso el
choque externo aporta el 83 % del tamaño de los choques condicionales y los domésticos el 17 %
restante: el modelo "explica" parte de la caída de los precios de exportación con sorpresas del PBI y
de la recaudación, algo poco creíble en una economía pequeña y abierta. La exogeneidad de bloque de la
sesión 2 apaga esas columnas.

Segundo, los escenarios **no son simétricos**, y no por el modelo, que es lineal: el adverso se aparta
de la trayectoria base mucho más que el favorable, porque la base ya esperaba precios de exportación
creciendo entre 10 y 18 %. Tercero, la **razón entre efectos** se mantiene: en ambos casos la
recaudación se mueve unas seis veces y media más que el PBI, coherente con el peso de la minería en
los ingresos fiscales.

## Ejercicios

1. Sortee los choques futuros con otra raíz de $\Sigma^{(m)}$, por ejemplo la simétrica
   $V\Lambda^{1/2}V'$, y verifique que la densidad predictiva no cambia.
2. Diseñe un escenario con la trayectoria **observada** de $x$ en 2018-2019. ¿Cuánto del PBI y de la
   recaudación observados explica?
3. Agregue exogeneidad de bloque (`b_exo`) y arme de nuevo $A$: ¿qué columnas se vuelven cero?
   ¿Cambia la lectura del escenario adverso? Es el punto de partida de la sesión 2.
4. Repita el escenario adverso con `shock_uncertainty=False` y compare el ancho de las bandas. ¿Qué
   incertidumbre desaparece?